In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [2]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
 
    data = np.asarray(data)

    boot_samples = np.random.choice(data, size=(n_boot, len(data)), replace=True)
    
    boot_stats = stat_func(boot_samples, axis=1)
    
    alpha = 1 - ci_level
    lower = np.percentile(boot_stats, 100 * (alpha / 2))
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    
    return (lower, upper)

data_test = np.arange(1, 101)
np.random.seed(42) 
ci_result = bootstrap_ci(data_test)

print(f"95% CI: {ci_result}")

95% CI: (np.float64(44.799749999999996), np.float64(56.100249999999996))


In [4]:

n = 250

continuous_data = np.random.normal(loc=150, scale=25, size=n)
binary_data = np.random.binomial(n=1, p=0.60, size=n)        

df = pd.DataFrame({
    'response_time': continuous_data,
    'passed': binary_data
})

mean_ci = bootstrap_ci(df['response_time'], stat_func=np.mean)
median_ci = bootstrap_ci(df['response_time'], stat_func=np.median)
prop_ci = bootstrap_ci(df['passed'], stat_func=np.mean)

task2_summary = pd.DataFrame({
    'Statistic': ['Mean', 'Median', 'Proportion'],
    'Column': ['response_time', 'response_time', 'passed'],
    'stat_func': ['np.mean', 'np.median', 'np.mean'],
    'Bootstrap Lower': [mean_ci[0], median_ci[0], prop_ci[0]],
    'Bootstrap Upper': [mean_ci[1], median_ci[1], prop_ci[1]]
})


print(task2_summary.to_markdown(index=False))
print("\n")

| Statistic   | Column        | stat_func   |   Bootstrap Lower |   Bootstrap Upper |
|:------------|:--------------|:------------|------------------:|------------------:|
| Mean        | response_time | np.mean     |           148.625 |           155.105 |
| Median      | response_time | np.median   |           144.934 |           157.449 |
| Proportion  | passed        | np.mean     |             0.532 |             0.652 |




In [5]:
mean_est = df['response_time'].mean()
se_mean = st.sem(df['response_time'])
norm_mean_ci = st.t.interval(confidence=0.95, df=n-1, loc=mean_est, scale=se_mean)

median_est = df['response_time'].median()
se_median = 1.2533 * se_mean 
norm_median_ci = st.t.interval(confidence=0.95, df=n-1, loc=median_est, scale=se_median)

p_hat = df['passed'].mean()
z = 1.96
se_prop = np.sqrt((p_hat * (1 - p_hat)) / n)
wald_lower = p_hat - z * se_prop
wald_upper = p_hat + z * se_prop
norm_prop_ci = (wald_lower, wald_upper)

task3_comparison = pd.DataFrame({
    'Statistic': ['Mean', 'Median', 'Proportion'],
    'Bootstrap Lower': [mean_ci[0], median_ci[0], prop_ci[0]],
    'Bootstrap Upper': [mean_ci[1], median_ci[1], prop_ci[1]],
    'Normal Lower': [norm_mean_ci[0], norm_median_ci[0], norm_prop_ci[0]],
    'Normal Upper': [norm_mean_ci[1], norm_median_ci[1], norm_prop_ci[1]]
})


print(task3_comparison.to_markdown(index=False))

| Statistic   |   Bootstrap Lower |   Bootstrap Upper |   Normal Lower |   Normal Upper |
|:------------|------------------:|------------------:|---------------:|---------------:|
| Mean        |           148.625 |           155.105 |     148.669    |     155.158    |
| Median      |           144.934 |           157.449 |     144.902    |     153.035    |
| Proportion  |             0.532 |             0.652 |       0.531078 |       0.652922 |


These two methods are almost identical in practice. When it comes to estimating the median, the bootstrap technique is especially valuable because, unlike the mean or proportion, there isn’t a standard closed‑form formula for the standard error of the sample median. If the data are not normally distributed, relying on traditional confidence interval assumptions without bootstrap often leads to unreliable results.